# Bifrost Getting Started — Groq + Mistral + Jina + Qdrant Cloud

### A free-tier, self-hosted AI gateway walkthrough

---

**Purpose:** Get hands-on with [Bifrost](https://github.com/maximhq/bifrost) (self-hosted, open-source AI
gateway by Maxim AI) using only providers with usable free tiers:

- **Groq** — chat/completions (LPU inference, very fast, generous free tier)
- **Mistral** — chat/completions (La Plateforme free tier)
- **Jina AI** — embeddings (used only in the RAG section, called directly — not via Bifrost)
- **Qdrant Cloud** — vector store (managed, used only in the RAG section)

**Your setup:** a single, already-created Docker container named `bifrost`, started with
`docker start bifrost` and nothing else. Everything Bifrost-side is configured through its
**Web UI** at `http://localhost:8080`, which persists inside that container across restarts.

**What you'll cover:**
1. Baseline: vanilla Groq/Mistral SDKs (no gateway)
2. Bifrost setup via the Web UI & a drop-in client
3. Bifrost feature showcase: unified API, failover, load balancing, streaming, observability, virtual keys, MCP
4. A small RAG pipeline built by hand (Jina embeddings + Qdrant Cloud + Bifrost generation)

**Prerequisites:**
1. Python 3.10+ and packages from `requirements.txt` (`pip install -r requirements.txt`)
2. `.env` filled in — `GROQ_API_KEY`, `MISTRAL_API_KEY` required from Section 1 onward; `JINA_API_KEY`,
   `QDRANT_URL`, and `QDRANT_API_KEY` required only for Section 4
3. `docker start bifrost` — that's it, nothing else to run
4. Groq + Mistral added as providers in the Bifrost Web UI (Section 2.2 walks through this)

**A note on style:** every code cell here does one small thing. Where the same handful of lines
would otherwise repeat across features (timing a call, reading which provider answered), there's
one small helper function instead — defined once, reused everywhere. Full term definitions live
in [`docs/00-terminologies.md`](docs/00-terminologies.md).

---


---
<a id="section-0"></a>
## Section 0: Environment Setup & Prerequisites


### 0.1 Install Dependencies

Installs everything this notebook needs. Run once per environment.


In [ ]:
%%capture
!pip install -r requirements.txt --quiet


### 0.2 Load Environment Variables

Reads your keys from `.env`. `GROQ_API_KEY` and `MISTRAL_API_KEY` are required from Section 1
onward, so we fail fast here with a clear message instead of hitting a confusing error later.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
BIFROST_BASE_URL = os.getenv("BIFROST_BASE_URL", "http://localhost:8080")
BIFROST_GROQ_VIRTUAL_KEY = os.getenv("BIFROST_GROQ_VIRTUAL_KEY", "")
BIFROST_MISTRAL_VIRTUAL_KEY = os.getenv("BIFROST_MISTRAL_VIRTUAL_KEY", "")

assert GROQ_API_KEY, "Missing GROQ_API_KEY — set it in .env"
assert MISTRAL_API_KEY, "Missing MISTRAL_API_KEY — set it in .env"

print("Keys loaded. Bifrost base URL:", BIFROST_BASE_URL)


### 0.3 Bifrost Health Check

A quick check that the `bifrost` container is actually up before anything else tries to call it.


In [ ]:
import httpx

try:
    health = httpx.get(f"{BIFROST_BASE_URL}/health", timeout=3)
    bifrost_ok = health.status_code == 200
except Exception:
    bifrost_ok = False

print("Bifrost:", "Online" if bifrost_ok else "Offline")

if not bifrost_ok:
    print("Start it with: docker start bifrost")


### 0.4 Prompts We'll Reuse

One set of prompts, used throughout the notebook, so the vanilla SDKs, Bifrost, and the RAG
section are all answering the same questions.


In [ ]:
TEST_PROMPTS = {
    "simple": "What is the capital of France?",
    "reasoning": "Explain the difference between RAG and fine-tuning in 3 bullet points.",
    "code": "Write a Python function that validates an email address using regex.",
    "duplicate1": "What is LangChain used for?",
    "duplicate2": "What is LangChain primarily used for?",
    "deepwiki": "What are the stream modes in the new langgraph version? Use the deepwiki tool to check the langchain-ai/langgraph repo.",
    "tavily": "Search the web for the latest news about Groq AI and summarize the top result.",
}


### 0.5 Models We'll Use

Groq is retiring `llama-3.3-70b-versatile` and `llama-3.1-8b-instant` on 2026-08-16 (free tier),
so this notebook uses their recommended replacements instead. Mistral's `-latest` aliases always
point at the current model, so they need no such swap.

`openai/gpt-oss-120b` is OpenAI's *open-weight* model, served on Groq's own hardware — it runs on
`GROQ_API_KEY`, not an OpenAI key. Nothing here touches your OpenAI account.


In [ ]:
import time

GROQ_MODEL = "openai/gpt-oss-120b"            # capable — replaces llama-3.3-70b-versatile
GROQ_MODEL_FALLBACK = "openai/gpt-oss-20b"    # fast/cheap — replaces llama-3.1-8b-instant
MISTRAL_MODEL = "mistral-large-latest"
MISTRAL_MODEL_FALLBACK = "mistral-small-latest"

print("Groq:", GROQ_MODEL, "| fallback:", GROQ_MODEL_FALLBACK)
print("Mistral:", MISTRAL_MODEL, "| fallback:", MISTRAL_MODEL_FALLBACK)


---
<a id="section-1"></a>
## Section 1: Baseline — Vanilla Groq & Mistral SDKs

> **Goal:** show what you have to write yourself WITHOUT a gateway — two separate SDKs, two
> separate error-handling paths, no shared observability.


### 1.1 Vanilla Groq SDK — Basic Chat Completion

Calls Groq directly, no gateway involved. This is the reference point every later section gets
compared against.


In [ ]:
from groq import Groq

client_groq = Groq(api_key=GROQ_API_KEY)


In [ ]:
start = time.perf_counter()
response = client_groq.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": TEST_PROMPTS["simple"]}],
)
latency_ms = (time.perf_counter() - start) * 1000

print(response.choices[0].message.content)
print(f"\n{latency_ms:.0f} ms, {response.usage.total_tokens} tokens")


### 1.2 Vanilla Mistral SDK — Basic Chat Completion

Same idea, on Mistral's own SDK. Notice the different client shape (`chat.complete` vs
`chat.completions.create`) — this per-provider inconsistency is exactly what a gateway hides.

> In mistralai SDK 2.x, the import moved to `mistralai.client` — `from mistralai import Mistral`
> now raises `ImportError`.


In [ ]:
from mistralai.client import Mistral

client_mistral = Mistral(api_key=MISTRAL_API_KEY)


In [ ]:
start = time.perf_counter()
response = client_mistral.chat.complete(
    model=MISTRAL_MODEL,
    messages=[{"role": "user", "content": TEST_PROMPTS["reasoning"]}],
)
latency_ms = (time.perf_counter() - start) * 1000

print(response.choices[0].message.content)
print(f"\n{latency_ms:.0f} ms, {response.usage.total_tokens} tokens")


### 1.3 Vanilla Streaming — Groq

For chat UIs that render tokens as they arrive, instead of waiting for the full response. Shown
here on the raw SDK so Section 3.4 can show the same thing through Bifrost.


In [ ]:
stream = client_groq.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": TEST_PROMPTS["code"]}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()


### 1.4 What's Missing Without a Gateway

| Problem                   | Vanilla Groq SDK | Vanilla Mistral SDK |
|---------------------------|-------------------|-----------------------|
| Automatic failover        | Manual            | Manual                |
| Multi-provider routing    | Not possible      | Not possible           |
| Unified request/response  | Per-SDK           | Per-SDK                |
| Rate limit handling       | Manual retry      | Manual retry           |
| Observability / tracing   | External only     | External only          |
| Budget enforcement        | Not possible      | Not possible            |
| MCP tool governance        | Not possible      | Not possible            |
| Streaming normalization   | Per-provider       | Per-provider            |


### 1.5 Manual Fallback — What You'd Have to Write Yourself

The fallback logic teams end up hand-writing without a gateway: try Groq, and if it fails, retry
on Mistral. Section 3.2 replaces this whole function with a single HTTP header.


In [ ]:
def naive_manual_fallback(prompt: str) -> str:
    try:
        response = client_groq.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content
    except Exception as groq_error:
        print("Groq failed, trying Mistral instead:", groq_error)
        response = client_mistral.chat.complete(
            model=MISTRAL_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content


In [ ]:
answer = naive_manual_fallback(TEST_PROMPTS["simple"])
print(answer)


---
<a id="section-2"></a>
## Section 2: Bifrost Setup & Architecture

> **Bifrost** is a Go-based, open-source AI gateway. It provides failover, load balancing,
> virtual keys, an MCP gateway, and observability — all via a single self-hosted container.


### 2.1 Your Bifrost Container

You already have a container named `bifrost`. The full lifecycle for this notebook is just:

```bash
docker start bifrost   # bring it up
docker stop bifrost    # shut it down when you're done
```

No `docker run`, no `docker-compose`, no extra containers. Bifrost persists whatever you
configure through its Web UI **inside that same container**, so it survives stop/start cycles.

**Do you need a Bifrost API key?** No. Bifrost doesn't require its own credential for incoming
requests by default — that's Virtual Keys (Section 3.7), an optional layer you can turn on later.
The client below is the *unscoped* drop-in client: it needs to reach both Groq and Mistral, so it
uses a placeholder instead of a virtual key (a virtual key is scoped to whatever provider it was
created for — Section 3.7 uses `BIFROST_GROQ_VIRTUAL_KEY` and `BIFROST_MISTRAL_VIRTUAL_KEY`
directly, one client per provider).


### 2.2 Add Groq & Mistral as Providers (Web UI)

1. Open `http://localhost:8080`
2. Go to **Providers** → **Add Provider**
3. Provider = `groq`, paste your actual `GROQ_API_KEY` value directly into the key field
4. Repeat for provider = `mistral` with your `MISTRAL_API_KEY`
5. (Optional, for Section 3.8) Go to **MCP Gateway → MCP Catalog** → **Add MCP Server**:
   - `deepwiki` — connection type `HTTP (Streamable)`, URL `https://mcp.deepwiki.com/mcp`, auth `None`
   - `tavily` — same connection type, URL `https://mcp.tavily.com/mcp/?tavilyApiKey=YOUR_TAVILY_KEY`
     (free key from `https://app.tavily.com/home`), auth `None`
   - **Important:** open each one and toggle **Auto-execute** on for its tools. Without this,
     Bifrost proposes the tool call and stops — it won't actually run it.


### 2.3 Bifrost Drop-in Client Setup

One `OpenAI` client instance now talks to every provider Bifrost knows about — which provider
handles a call is picked by the `"<provider>/<model>"` string, not by which client you made.


In [ ]:
from openai import OpenAI

bifrost = OpenAI(
    api_key="not-needed-unless-virtual-keys-enabled",
    base_url=f"{BIFROST_BASE_URL}/v1",
)

print("Bifrost client ready. Pick a provider via the model string, e.g.:")
print(f"  groq/{GROQ_MODEL}")
print(f"  mistral/{MISTRAL_MODEL}")


---
<a id="section-3"></a>
## Section 3: Bifrost Feature Showcase


### 3.0 One Helper We'll Reuse

Every feature below calls Bifrost the same way: send a prompt, get back the answer plus *which
provider* actually handled it. Bifrost attaches that provider info as an extra field on the
response. The OpenAI SDK keeps unknown fields instead of dropping them, so we can read it
straight off `response.model_extra` — no manual response parsing needed.


In [ ]:
def call_bifrost(prompt: str, model: str, fallback_model: str = None):
    headers = {"x-bifrost-fallback-models": fallback_model} if fallback_model else None

    response = bifrost.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        extra_headers=headers,
    )

    provider = response.model_extra.get("extra_fields", {}).get("provider", "unknown")
    return response.choices[0].message.content, provider


### 3.1 Feature 1 — Unified Drop-in API (Groq + Mistral, Same Client)

Swap providers per-request — cheap/fast Groq for simple queries, Mistral for others — without
maintaining two SDKs or two code paths.


In [ ]:
answer, provider = call_bifrost(TEST_PROMPTS["simple"], f"groq/{GROQ_MODEL}")
print(f"[{provider}] {answer}")


In [ ]:
answer, provider = call_bifrost(TEST_PROMPTS["simple"], f"mistral/{MISTRAL_MODEL}")
print(f"[{provider}] {answer}")


### 3.2 Feature 2 — Automatic Failover / Provider Fallback

Keep serving requests even if your primary provider is down or rate-limiting you. Bifrost retries
on the fallback model automatically — your code never sees the failure. It triggers on **5xx,
429, and 401**; it does **not** trigger on a **404** (model not found) — that's a bug in your
request, not an outage, so Bifrost passes the error straight through instead of masking it.


In [ ]:
# Normal call — Groq is healthy, so the fallback never kicks in
answer, provider = call_bifrost(
    TEST_PROMPTS["simple"], f"groq/{GROQ_MODEL}", fallback_model=f"mistral/{MISTRAL_MODEL}"
)
print(f"[{provider}] {answer}")


In [ ]:
# Bad model name (404) — does NOT trigger the Mistral fallback, by design
try:
    answer, provider = call_bifrost(
        TEST_PROMPTS["simple"], "groq/not-a-real-model", fallback_model=f"mistral/{MISTRAL_MODEL}"
    )
    print(f"[{provider}] {answer}")
except Exception as e:
    print("No fallback triggered, as expected for a 404:", e)


### 3.3 Feature 3 — Weighted Load Balancing Across API Keys

Spread traffic across multiple keys for the *same* provider to raise your effective rate limit.
Add a second key under the `groq` provider in the Web UI, with its own weight, to see traffic
split. With only one key configured (the default here), every request just uses that one key.


In [ ]:
answer, provider = call_bifrost("Hello", f"groq/{GROQ_MODEL}")
print("Provider used:", provider)


### 3.4 Feature 4 — Streaming (Unified Across Providers)

Same streaming function, no `if provider == "groq"` branching — swap the model string and the
same code streams from a different backend.


In [ ]:
def stream_bifrost(prompt: str, model: str):
    stream = bifrost.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
    print()


In [ ]:
stream_bifrost(TEST_PROMPTS["code"], f"groq/{GROQ_MODEL}")


In [ ]:
stream_bifrost(TEST_PROMPTS["code"], f"mistral/{MISTRAL_MODEL}")


### 3.5 Feature 5 — Observability: Logs API

See exactly what Bifrost routed, to which provider, and how long it took — without adding any
logging of your own. Full Prometheus `/metrics` needs a separate plugin binary (not covered
here); this REST endpoint works out of the box.


In [ ]:
import pprint

response = httpx.get(f"{BIFROST_BASE_URL}/api/logs?limit=5", timeout=5)
logs = response.json() if response.status_code == 200 else {"status": response.status_code}

pprint.pprint(logs)


### 3.6 Feature 6 — OpenTelemetry Tracing (Optional)

Useful when Bifrost is one hop in a bigger system. Needs an OTEL collector (e.g. Jaeger) and
Bifrost's `otel` plugin pointed at it — infra we're not standing up here. The call below still
succeeds either way; there's just nowhere to view the trace. Feel free to skip this cell.


In [ ]:
response = bifrost.chat.completions.create(
    model=f"groq/{GROQ_MODEL}",
    messages=[{"role": "user", "content": "Say hello and confirm you received this message."}],
    extra_headers={"x-bifrost-trace-id": "notebook-demo-001"},
)

print(response.choices[0].message.content)


### 3.7 Feature 7 — Virtual Keys & Budget Governance

Hand a teammate, or a specific app, a scoped key with a spending cap — instead of giving out your
raw Groq/Mistral keys. Create one at `http://localhost:8080` → **Virtual Keys** (a separate tab
from **Providers**) for each provider you want to govern separately. Bifrost shows you the
generated secret once — put it in `.env` as `BIFROST_GROQ_VIRTUAL_KEY` /
`BIFROST_MISTRAL_VIRTUAL_KEY` to try these cells. Each key only works for the provider it was
created for, so unlike Section 2.3's client, this is one client per virtual key.


In [ ]:
def call_with_virtual_key(virtual_key: str, model: str, prompt: str):
    if not virtual_key:
        print("No virtual key set for this provider — create one at http://localhost:8080 -> Virtual Keys")
        return
    client = OpenAI(api_key=virtual_key, base_url=f"{BIFROST_BASE_URL}/v1")
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        print(response.choices[0].message.content)
    except Exception as e:
        print("Call failed — check the virtual key is valid and not over budget:", e)


In [ ]:
call_with_virtual_key(BIFROST_GROQ_VIRTUAL_KEY, f"groq/{GROQ_MODEL}", TEST_PROMPTS["simple"])


In [ ]:
call_with_virtual_key(BIFROST_MISTRAL_VIRTUAL_KEY, f"mistral/{MISTRAL_MODEL}", TEST_PROMPTS["simple"])


### 3.8 Feature 8 — MCP Gateway (DeepWiki + Tavily)

Let the model call external tools — DeepWiki (ask questions about a GitHub repo) and Tavily (live
web search) — through the gateway, instead of wiring tool-calling into every client app
separately. Requires both MCP clients added with **Auto-execute** on (Section 2.2).

> **Model note:** Groq's `openai/gpt-oss-*` models fail with a 400 during Bifrost's tool-calling
> loop (`'reasoning_content' is unsupported` — Bifrost resends the model's own reasoning trace,
> and Groq's API rejects it for these models). So this section runs on **Mistral** instead, which
> has no such issue.


In [ ]:
# What MCP servers does Bifrost actually have registered?
response = httpx.get(f"{BIFROST_BASE_URL}/api/mcp/clients", timeout=5)
clients = response.json().get("clients", [])
registered = {c["config"]["name"] for c in clients}

print("Registered:", registered or "none")


In [ ]:
def call_with_tools(prompt: str, servers: list[str]) -> dict:
    response = httpx.post(
        f"{BIFROST_BASE_URL}/v1/chat/completions",
        json={
            "model": f"mistral/{MISTRAL_MODEL}",
            "messages": [{"role": "user", "content": prompt}],
            "bifrost": {"mcp": {"enabled": True, "code_mode": True, "servers": servers}},
        },
        timeout=60,
    )
    return response.json()["choices"][0]["message"]


In [ ]:
# DeepWiki — ask a question about a GitHub repo
if "deepwiki" in registered:
    message = call_with_tools(TEST_PROMPTS["deepwiki"], ["deepwiki"])
    print(message.get("content") or message.get("tool_calls"))
else:
    print("Skipped — add the deepwiki MCP server first (see Section 2.2)")


In [ ]:
# Tavily — live web search
if "tavily" in registered:
    message = call_with_tools(TEST_PROMPTS["tavily"], ["tavily"])
    print(message.get("content") or message.get("tool_calls"))
else:
    print("Skipped — add the tavily MCP server first (see Section 2.2)")


> **Not covered here** (need paid/enterprise infra): Guardrails (AWS Bedrock / Azure Content
> Safety), Cluster Mode (multiple Bifrost nodes), Secret Manager integration (HashiCorp Vault /
> cloud secret managers). See [`docs/03-oss-vs-enterprise.md`](docs/03-oss-vs-enterprise.md).


---
<a id="section-4"></a>
## Section 4: A Small RAG Pipeline (Jina Embeddings + Qdrant Cloud + Bifrost Generation)

> Built by hand, not through Bifrost's semantic-cache plugin, so every step (embed, store,
> retrieve, generate) is visible. Generation still goes through Bifrost; only embeddings (Jina)
> and vector search (Qdrant Cloud) sit outside the gateway.


### 4.1 Connect to Qdrant Cloud

A managed, always-on vector store you don't have to run yourself — just a URL and an API key
from your Qdrant Cloud cluster dashboard.


In [ ]:
JINA_API_KEY = os.getenv("JINA_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

assert JINA_API_KEY, "Missing JINA_API_KEY — set it in .env"
assert QDRANT_URL, "Missing QDRANT_URL — set it in .env"
assert QDRANT_API_KEY, "Missing QDRANT_API_KEY — set it in .env"


In [ ]:
# port=443 because Qdrant Cloud serves REST over standard HTTPS —
# the client's default port (6333) gets connection-reset on a cloud cluster.
from qdrant_client import QdrantClient

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, port=443)
print("Connected. Existing collections:", [c.name for c in qdrant.get_collections().collections])


### 4.2 Jina Embeddings Helper

Turns text into vectors for semantic search. `task="retrieval.passage"` is tuned for documents
you're storing; `task="retrieval.query"` is tuned for the question you're searching with.


In [ ]:
import requests

JINA_MODEL = "jina-embeddings-v4"

def jina_embed(texts: list[str], task: str = "retrieval.passage") -> list[list[float]]:
    response = requests.post(
        "https://api.jina.ai/v1/embeddings",
        headers={"Authorization": f"Bearer {JINA_API_KEY}"},
        json={"model": JINA_MODEL, "input": texts, "task": task},
        timeout=30,
    )
    response.raise_for_status()
    return [item["embedding"] for item in response.json()["data"]]


In [ ]:
# Read the dimension from a live call instead of hardcoding it —
# a wrong hardcoded dimension is a common Bifrost + Qdrant mistake.
test_vector = jina_embed(["hello world"])
JINA_DIM = len(test_vector[0])
print("Embedding dimension:", JINA_DIM)


### 4.3 Build a Tiny Knowledge Base in Qdrant Cloud

Seeds the vector store with a handful of reference documents so the retrieval step in 4.4 has
something real to find. Re-running this wipes and rebuilds the demo collection.


In [ ]:
from qdrant_client.models import Distance, VectorParams, PointStruct

COLLECTION = "bifrost_rag_demo"

DOCS = [
    "Bifrost is a Go-based, open-source AI gateway built by Maxim AI, released under Apache 2.0.",
    "Bifrost supports automatic failover across providers, triggered on 5xx errors, 429 rate limits, and 401 auth failures.",
    "Bifrost exposes a single OpenAI-compatible /v1/chat/completions endpoint; the provider is selected via a \"provider/model\" string.",
    "Groq serves LLMs on custom LPU hardware, offering very low-latency inference with a free API tier.",
    "Mistral AI offers La Plateforme with a free tier for models like mistral-large-latest and mistral-small-latest.",
    "Qdrant Cloud is a managed vector database used here for RAG retrieval, reached over a URL + API key.",
    "Jina AI provides an embeddings API (jina-embeddings-v4) with task-specific adapters for retrieval and classification.",
]

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=JINA_DIM, distance=Distance.COSINE),
)
print("Collection ready:", COLLECTION)


In [ ]:
vectors = jina_embed(DOCS)

qdrant.upsert(
    collection_name=COLLECTION,
    points=[
        PointStruct(id=i, vector=vector, payload={"text": doc})
        for i, (vector, doc) in enumerate(zip(vectors, DOCS))
    ],
)

print(f'Indexed {len(DOCS)} documents into "{COLLECTION}"')


### 4.4 Retrieve + Generate (Full RAG Loop, Generation via Bifrost)

The actual RAG pattern: embed the question, retrieve the closest stored facts from Qdrant, then
ask an LLM — through Bifrost, using the same `call_bifrost` helper from Section 3 — to answer
using only that retrieved context.


In [ ]:
def rag_query(question: str, model: str = f"groq/{GROQ_MODEL}", top_k: int = 3) -> dict:
    query_vector = jina_embed([question], task="retrieval.query")[0]

    hits = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        limit=top_k,
    ).points

    context = "\n".join(f"- {hit.payload['text']}" for hit in hits)
    prompt = (
        "Answer the question using ONLY the context below. "
        "If the answer isn't in the context, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )

    answer, provider = call_bifrost(prompt, model)
    return {"retrieved": hits, "answer": answer, "provider": provider}


In [ ]:
result = rag_query("What hardware does Groq use for inference?")

print("Retrieved context:")
for hit in result["retrieved"]:
    print(f"  [{round(hit.score, 3)}] {hit.payload['text']}")

print("\nAnswer:", result["answer"])
print("Provider:", result["provider"])


---
## Summary

| Covered | Skipped (needs paid/enterprise infra) |
|---|---|
| Vanilla Groq/Mistral SDK baseline | Guardrails (AWS Bedrock / Azure) |
| Drop-in unified Bifrost client (Web UI configured) | Cluster mode (multi-node) |
| Automatic failover, weighted load balancing | Secret Manager (Vault / cloud) |
| Unified streaming | Prometheus `/metrics` (needs plugin binary) |
| Logs API, optional OpenTelemetry tracing | |
| Virtual keys & budgets | |
| MCP gateway (DeepWiki + Tavily) | |
| Hand-built RAG: Jina embeddings + Qdrant Cloud + Bifrost generation | |

For a full breakdown of every term used above, see [`docs/00-terminologies.md`](docs/00-terminologies.md).
For the enterprise-tier features, see [`docs/03-oss-vs-enterprise.md`](docs/03-oss-vs-enterprise.md).
